# Targeted Marketing Optimization & Uplift Strategy (Starbucks Dataset)
## Transforming Mass-Blast Promotions into AI-Driven Causal Uplift Targeting

### Executive Summary
In traditional promotional marketing, offers are blast-broadcasted to all members regardless of customer responsiveness. This incurs substantial subsidy waste on customers who would purchase anyway (*Sure Things / Cannibalization*) and causes fatigue among non-responsive users.

This project implements an end-to-end Machine Learning pipeline that:
1. **Causally aligns the event funnel**: Validates that an offer was seen before being completed.
2. **Eliminates wasteful reward subsidies**: Distinguishes authentic conversions from accidental completions.
3. **Predicts calibrated propensity scores**: Leverages demographic, promo metadata, and customer historical behaviors.
4. **Optimizes unit economics**: Generates Cumulative Lift & Gain curves to determine the profit-maximizing audience cutoff.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)

# Add root to sys.path
ROOT_DIR = os.path.dirname(os.path.abspath('')) if 'notebooks' in os.getcwd() else os.getcwd()
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from src.data_loader import load_all_raw_data
from src.funnel_builder import align_customer_offer_funnel
from src.feature_engineering import build_features
from src.model import split_by_customer, train_calibrated_model, evaluate_and_score, compute_decile_table
from src.financial_sim import run_financial_simulation, plot_and_save_visualizations

print('Libraries and custom modules loaded successfully!')

## 1. Data Ingestion & Preprocessing
We ingest the three relational Starbucks files: `portfolio.json`, `profile.json`, and `transcript.json`.
Special handling is performed for:
- Demographic missingness (encoded as age `118`, which directly correlates with null income and gender).
- Converting member signup dates into customer `tenure_days`.
- Unpacking the event log dictionary column `value` into `offer_id`, `amount`, and `reward`.

In [ ]:
data_dir = os.path.join(ROOT_DIR, 'data', 'raw')
portfolio, profile, transcript = load_all_raw_data(data_dir)

print(f'Portfolio offers:    {len(portfolio)}')
print(f'Customer profiles:   {len(profile):,} (Missing demographics: {profile["demographics_missing"].sum():,})')
print(f'Transcript events:   {len(transcript):,}')

display(portfolio.head(3))
display(profile.head(3))

## 2. Causal Funnel Alignment & Target Definition
A critical flaw in standard analysis is treating any `offer completed` event as marketing success.
- **Valid Conversion ($y = 1$):** Customer receives offer $\rightarrow$ views offer $\rightarrow$ transacts and completes offer within validity duration ($view\_time \le complete\_time$).
- **Wasteful Conversion ($y = 0$):** Customer completes offer without ever viewing it (*Sure Things* / cannibalized organic spend).
- **Non-Conversion ($y = 0$):** Customer viewed or ignored offer, but it expired incomplete.

In [ ]:
funnel_df = align_customer_offer_funnel(transcript, portfolio)

print('=== FUNNEL CATEGORIES BREAKDOWN ===')
display(funnel_df['funnel_category'].value_counts())

# Measure wasteful vs valid subsidies
reward_stats = funnel_df.groupby('funnel_category')['actual_reward_incurred'].sum().reset_index()
reward_stats['pct_total_spend'] = (reward_stats['actual_reward_incurred'] / reward_stats['actual_reward_incurred'].sum()) * 100
display(reward_stats)

## 3. Feature Engineering
We construct interaction-level features combining demographic attributes, promotional attributes, and historical customer engagement behavior.

In [ ]:
features_df = build_features(funnel_df, profile, portfolio, transcript)
print(f'Feature matrix dimensions: {features_df.shape}')
features_df.head(3)

## 4. Model Training & Probability Calibration
We apply a Group Shuffle Split on `person_id` (80/20 train/test) to evaluate out-of-sample generalization.
We train a Gradient Boosting model with **Isotonic Calibration** to ensure predicted propensities accurately reflect true empirical conversion rates.

In [ ]:
train_df, test_df = split_by_customer(features_df, test_size=0.20, random_state=42)
print(f'Train instances: {len(train_df):,} | Test instances: {len(test_df):,}')

model = train_calibrated_model(train_df)
scored_test_df, metrics = evaluate_and_score(model, test_df)

## 5. Decile Performance & Business Simulation
We rank customer-offer interactions into 10 deciles (Decile 1 = top 10% highest conversion propensity).
We then simulate campaign economics comparing a **Mass Blast** approach against the **Optimal Decile Targeting** strategy.

In [ ]:
dec_df, comparison_summary = run_financial_simulation(
    scored_test_df,
    gross_margin_rate=0.60,
    cost_per_message=0.05
)

print('=== DECILE PERFORMANCE TABLE ===')
display(dec_df[['decile', 'customers', 'converters', 'conv_rate', 'decile_lift', 'cum_lift', 'revenue', 'reward_cost', 'net_profit', 'roi']])

print('\n=== EXECUTIVE SUMMARY COMPARISON ===')
display(comparison_summary)

## 6. Strategic Takeaways & Recommendations
- **Cost Avoidance**: Targeting the top deciles avoids spending promotional subsidies on unresponsive customers and self-converting buyers.
- **Net Margin Maximization**: Incremental ROI improves dramatically when trimming low-efficiency deciles (D7-D10).
- **Operational Implementation**: Deploy the propensity scoring model as a pre-scoring service prior to every promotional blast cycle.